# Rebuild All Figures One PNG At A Time

Generated by `Figures/rebuild_figures.py`. Each code cell below corresponds to exactly one grouped PNG in `Figures/manifest.csv`: it refreshes that figure's source artifact, copies that one PNG into `Figures/`, and displays that one image inline.

In [ ]:
import importlib.util
import math
import shutil
import sys
from pathlib import Path
from IPython.display import Image, display
import pandas as pd
import matplotlib.pyplot as plt

FIGURES_DIR = Path.cwd()
if FIGURES_DIR.name != 'Figures':
    FIGURES_DIR = Path('/home/bjyong/Complexity/Complexity/Figures')
ROOT = FIGURES_DIR.parent
IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg'}
MANIFEST = pd.read_csv(FIGURES_DIR / 'manifest.csv')

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

def _clear_local_import_cache():
    for name in list(sys.modules):
        if name == 'utils' or name.startswith('utils.') or name == 'make_summarized_outputs':
            sys.modules.pop(name, None)

def _load_script(script_rel):
    script = ROOT / script_rel
    if not script.exists():
        raise FileNotFoundError(script)
    module_name = '_single_fig_' + ''.join(ch if ch.isalnum() else '_' for ch in script_rel)
    spec = importlib.util.spec_from_file_location(module_name, script)
    module = importlib.util.module_from_spec(spec)
    old_sys_path = sys.path[:]
    sys.path.insert(0, str(script.parent))
    _clear_local_import_cache()
    try:
        assert spec.loader is not None
        sys.modules[module_name] = module
        spec.loader.exec_module(module)
    finally:
        _clear_local_import_cache()
        sys.path[:] = old_sys_path
    return module

def _manifest_row(category_path):
    rows = MANIFEST.loc[MANIFEST['category_path'].eq(category_path)]
    if rows.empty:
        raise KeyError(category_path)
    return rows.iloc[0]

def _split_inputs(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    text = str(value)
    if not text or text.lower() == 'nan':
        return []
    return [item for item in text.split(';') if item]

def _copy_display(source, target):
    if not source.exists():
        raise FileNotFoundError(source)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)
    print(f'source: {source.relative_to(ROOT).as_posix()}')
    print(f'output: {target.relative_to(ROOT).as_posix()}')
    display(Image(filename=str(target)))
    return target

def _generate_theory_source(source):
    rel = source.relative_to(ROOT).as_posix()
    if rel.startswith('01_theory/01_theory_analytic/figures/'):
        mod = _load_script('01_theory/01_theory_analytic/src/make_figures.py')
        mod.make_figure(mod.DEFAULT_INPUT_CSV, source)
        return
    if rel.endswith('01_theory/figures/fig01_sampling_vs_analytic_phi_by_distance_alpha0p1.png'):
        mod = _load_script('01_theory/figures/src/make_figures.py')
        mod.write_combined_figure(mod.read_required_csv(mod.DEFAULT_ANALYTIC_CSV), mod.read_sampling_input(mod.DEFAULT_SAMPLING_INPUT), source)
        return
    if rel.endswith('01_theory/02_theory_sampling/figures/phi_by_sampling/phi_by_sampling.png'):
        mod = _load_script('01_theory/02_theory_sampling/src/make_figures.py')
        mod.make_phi_figure(mod.DEFAULT_PHI_INPUT_ROOT, source)
        return
    if '01_theory/02_theory_sampling/figures/logZ_split_distributions/' in rel:
        mod = _load_script('01_theory/02_theory_sampling/src/make_figures.py')
        mod.plot_logz_split_distribution(mod.DEFAULT_LOGZ_INPUT_ROOT / f'{source.stem}.csv', source)
        return
    raise NotImplementedError(rel)

def _generate_synthetic_source(source):
    rel = source.relative_to(ROOT / '02_dnn_synthetic' / 'figures').as_posix()
    mod = _load_script('02_dnn_synthetic/figures/src/make_figures.py')
    if rel in {'01_dataset/sample_figure.png', '01_dataset/spin_dynamics_phase_transition.png'}:
        mod.build_dataset_figures()
        return
    if rel == '02_complexity_measure/beta_complexity_figure.png':
        mod.build_complexity_figure()
        return
    if rel.startswith('04_sampling/logZ_split_distributions/'):
        input_csv = mod.SAMPLING_INPUT_ROOT / f'{source.stem}.csv'
        mod.plot_logz_split_distribution(input_csv, source, max_scatter_per_radius=120)
        return
    if rel.startswith('05_proxy_local_entropy/'):
        for name, value_key, sem_key, ylabel, title, output_name in mod.CURVE_FIGURES:
            if source.name == output_name:
                input_csv = mod.PLE_INPUT_ROOT / name / f'{name}.csv'
                mod.plot_curve_frame(pd.read_csv(input_csv), value_key, sem_key, ylabel, title, source)
                return
        if source.name == 'phase_like_A_by_beta.png':
            phase_csv = mod.PLE_INPUT_ROOT / 'phase_like_A_by_beta' / 'phase_like_A_by_beta.csv'
            curve_csv = mod.PLE_INPUT_ROOT / 'phase_like_A_by_beta' / 'phase_derivative_curves.csv'
            mod.plot_phase_panel(pd.read_csv(phase_csv), pd.read_csv(curve_csv), 'beta', r'$\beta$', 'A measure by beta', source)
            return
        if source.name == 'phase_like_A_by_complexity.png':
            phase_csv = mod.PLE_INPUT_ROOT / 'phase_like_A_by_complexity' / 'phase_like_A_by_complexity.csv'
            curve_csv = mod.PLE_INPUT_ROOT / 'phase_like_A_by_complexity' / 'phase_derivative_curves.csv'
            mod.plot_phase_panel(pd.read_csv(phase_csv), pd.read_csv(curve_csv), 'complexity_mean', '3-NN complexity', 'A measure by complexity', source)
            return
    raise NotImplementedError(rel)

def _generate_mnist_stage_source(source):
    rel = source.relative_to(ROOT).as_posix()
    if rel.startswith('03_dnn_mnist/label_noise_sweep/01_dataset/figures/'):
        _load_script('03_dnn_mnist/label_noise_sweep/01_dataset/src/make_figures.py').main(); return
    if rel.startswith('03_dnn_mnist/manual_rules/01_dataset/figures/'):
        _load_script('03_dnn_mnist/manual_rules/01_dataset/src/make_figures.py').main(); return
    if rel.startswith('03_dnn_mnist/label_noise_sweep/02_complexity_measure/figures/'):
        mod = _load_script('03_dnn_mnist/label_noise_sweep/02_complexity_measure/src/make_figures.py'); mod._plot(mod._read_summary(mod.SUMMARY_PATH)); return
    if rel.startswith('03_dnn_mnist/manual_rules/02_complexity_measure/figures/'):
        mod = _load_script('03_dnn_mnist/manual_rules/02_complexity_measure/src/make_figures.py'); mod._plot(mod._read_summary(mod.SUMMARY_PATH)); return
    if rel.startswith('03_dnn_mnist/label_noise_sweep/03_reference_search/figures/'):
        _load_script('03_dnn_mnist/label_noise_sweep/03_reference_search/src/make_figures.py').build_figures(); return
    if rel.startswith('03_dnn_mnist/manual_rules/03_reference_search/figures/'):
        _load_script('03_dnn_mnist/manual_rules/03_reference_search/src/make_figures.py').build_figures(); return
    if rel.startswith('03_dnn_mnist/label_noise_sweep/04_sampling/figures/'):
        mod = _load_script('03_dnn_mnist/label_noise_sweep/04_sampling/src/make_figures.py')
        if source.name in {'eta_reference_phi_energy.png', 'eta_reference_direct_derivative.png'}:
            summary = pd.read_csv(mod.DEFAULT_UNIT_SUMMARY_ROOT / 'eta_reference_phi_by_eta_radius.csv')
            mod.plot_phi(summary, source.parent)
            return
        if 'logZ_split_distributions' in rel:
            mod.plot_logz_inputs(mod.DEFAULT_LOGZ_INPUT_ROOT, source.parents[1], max_scatter_per_radius=30)
            return
    if rel.startswith('03_dnn_mnist/manual_rules/04_sampling/figures/fresh108_validation/'):
        if source.exists():
            return
    if rel.startswith('03_dnn_mnist/manual_rules/04_sampling/figures/'):
        mod = _load_script('03_dnn_mnist/manual_rules/04_sampling/src/make_figures.py')
        phi = pd.read_csv(mod.SUMMARY_ROOT / 'figure_inputs' / 'phi_by_rule_radius.csv')
        qc = pd.read_csv(mod.SUMMARY_ROOT / 'figure_inputs' / 'qc_diagnostics_by_rule_radius.csv')
        if source.name == 'manual_rule_phi_energy.png': mod.plot_phi_curves(phi, source); return
        if source.name == 'manual_rule_direct_derivative.png': mod.plot_direct_derivative(phi, source); return
        if source.name == 'manual_rule_logZ_split_heatmap.png': mod.plot_qc_heatmap(qc, source); return
        if source.name == 'manual_rule_ess_q05.png': mod.plot_ess_floor(qc, source); return
        if 'logZ_split_distributions' in rel:
            mod.plot_logz_split_distributions(mod.SUMMARY_ROOT / 'figure_inputs' / 'logZ_split', source.parent)
            return
    if rel.startswith('03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/figures/'):
        mod = _load_script('03_dnn_mnist/label_noise_sweep/05_proxy_local_entropy/src/make_figures.py')
        for name, value_key, sem_key, ylabel, title, output_name in mod.CURVE_FIGURES:
            if source.name == output_name:
                mod._plot_curve(mod._read_csv(mod.FIGURE_INPUT_ROOT / name / f'{name}.csv'), value_key, sem_key, ylabel, title, source)
                return
        if source.name == 'phase_like_A_by_eta.png': mod._plot_phase('phase_like_A_by_eta', 'eta', 'label noise eta', source.name); return
        if source.name == 'phase_like_A_by_complexity.png': mod._plot_phase('phase_like_A_by_complexity', 'nmstv', '3-NN MNIST complexity', source.name); return
        if source.name == 'logZ_split_qc_results.png': mod._plot_logz_qc(mod._read_csv(mod.FIGURE_INPUT_ROOT / 'logZ_split_qc_results' / 'logZ_split_qc_results.csv'), source); return
        if source.name == 'reference_variability_results.png': mod._plot_reference_variability(mod._read_csv(mod.FIGURE_INPUT_ROOT / 'reference_variability_results' / 'reference_variability_results.csv'), source); return
    if rel.startswith('03_dnn_mnist/manual_rules/05_proxy_local_entropy/figures/'):
        mod = _load_script('03_dnn_mnist/manual_rules/05_proxy_local_entropy/src/make_figures.py')
        mod.build_summarized_outputs()
        phi = pd.read_csv(mod.FIGURE_INPUT_ROOT / 'phi_d_curve' / 'phi_d_curve.csv')
        dphi = pd.read_csv(mod.FIGURE_INPUT_ROOT / 'derivative_phi_d_curve' / 'derivative_phi_d_curve.csv')
        if source.name == 'phi_d_curve.png': mod._plot_curves(phi, 'delta_phi_energy_unit_mean', 'delta_phi_energy_unit_sem', 'phi(d) - phi(d0)', 'MNIST manual-rule phi(d)', source); return
        if source.name == 'phi_energetic_d_curve.png': mod._plot_curves(phi, 'phi_energy_raw_mean', 'phi_energy_raw_sem', 'energetic phi(d)', 'MNIST manual-rule energetic phi(d)', source); return
        if source.name == 'derivative_phi_d_curve.png': mod._plot_curves(dphi, 'd_delta_phi_energy_direct_dd_unit_mean', 'd_delta_phi_energy_direct_dd_unit_sem', 'd phi / dd', 'MNIST manual-rule direct derivative of phi(d)', source); return
        if source.name == 'derivative_phi_energetic_d_curve.png': mod._plot_curves(dphi, 'd_phi_energy_direct_dd_unit_mean', 'd_phi_energy_direct_dd_unit_sem', 'energetic d phi / dd', 'MNIST manual-rule direct energetic derivative', source); return
        if source.name == 'phase_like_A_by_rule.png': mod._plot_phase('rule_order', 'manual-rule order', 'phase_like_A_by_rule'); return
        if source.name == 'phase_like_A_by_complexity.png': mod._plot_phase('nmstv_mean', '3-NN MNIST complexity', 'phase_like_A_by_complexity'); return
        if source.name == 'logZ_split_qc_results.png': mod._plot_logz(source); return
        if source.name == 'reference_variability_results.png': mod._plot_reference_variability(source); return
    if source.exists():
        print(f'using existing source: {rel}')
        return
    raise NotImplementedError(rel)

def _generate_source(source, input_paths):
    rel = source.relative_to(ROOT).as_posix()
    if rel.startswith('01_theory/'):
        _generate_theory_source(source)
        return source
    if rel.startswith('02_dnn_synthetic/figures/'):
        _generate_synthetic_source(source)
        return source
    if rel.startswith('03_dnn_mnist/figures/'):
        stage_inputs = [ROOT / item for item in input_paths if item.lower().endswith(tuple(IMAGE_SUFFIXES))]
        if stage_inputs:
            stage_source = stage_inputs[0]
            _generate_mnist_stage_source(stage_source)
            source.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(stage_source, source)
            return source
    if rel.startswith('03_dnn_mnist/'):
        _generate_mnist_stage_source(source)
        return source
    if source.exists():
        return source
    raise NotImplementedError(rel)

def build_single_figure(category_path):
    row = _manifest_row(category_path)
    source = ROOT / str(row['source_path'])
    target = FIGURES_DIR / str(row['category_path'])
    input_paths = _split_inputs(row.get('input_paths'))
    print(f'figure: {category_path}')
    generated = _generate_source(source, input_paths)
    return _copy_display(generated, target)

FIGURES_DIR


## 01. theory/01_theory_analytic/phi_by_analytic_solution_alpha0p1.png

Source: `01_theory/01_theory_analytic/figures/phi_by_analytic_solution_alpha0p1.png`

In [ ]:
build_single_figure('theory/01_theory_analytic/phi_by_analytic_solution_alpha0p1.png')


## 02. theory/02_theory_sampling/logZ_split_distributions/N_160.png

Source: `01_theory/02_theory_sampling/figures/logZ_split_distributions/N_160.png`

In [ ]:
build_single_figure('theory/02_theory_sampling/logZ_split_distributions/N_160.png')


## 03. theory/02_theory_sampling/logZ_split_distributions/N_320.png

Source: `01_theory/02_theory_sampling/figures/logZ_split_distributions/N_320.png`

In [ ]:
build_single_figure('theory/02_theory_sampling/logZ_split_distributions/N_320.png')


## 04. theory/02_theory_sampling/logZ_split_distributions/N_40.png

Source: `01_theory/02_theory_sampling/figures/logZ_split_distributions/N_40.png`

In [ ]:
build_single_figure('theory/02_theory_sampling/logZ_split_distributions/N_40.png')


## 05. theory/02_theory_sampling/logZ_split_distributions/N_80.png

Source: `01_theory/02_theory_sampling/figures/logZ_split_distributions/N_80.png`

In [ ]:
build_single_figure('theory/02_theory_sampling/logZ_split_distributions/N_80.png')


## 06. theory/02_theory_sampling/phi_by_sampling/phi_by_sampling.png

Source: `01_theory/02_theory_sampling/figures/phi_by_sampling/phi_by_sampling.png`

In [ ]:
build_single_figure('theory/02_theory_sampling/phi_by_sampling/phi_by_sampling.png')


## 07. theory/summary/fig01_sampling_vs_analytic_phi_by_distance_alpha0p1.png

Source: `01_theory/figures/fig01_sampling_vs_analytic_phi_by_distance_alpha0p1.png`

In [ ]:
build_single_figure('theory/summary/fig01_sampling_vs_analytic_phi_by_distance_alpha0p1.png')


## 08. dnn_synthetic/01_dataset/sample_figure.png

Source: `02_dnn_synthetic/figures/01_dataset/sample_figure.png`

In [ ]:
build_single_figure('dnn_synthetic/01_dataset/sample_figure.png')


## 09. dnn_synthetic/01_dataset/spin_dynamics_phase_transition.png

Source: `02_dnn_synthetic/figures/01_dataset/spin_dynamics_phase_transition.png`

In [ ]:
build_single_figure('dnn_synthetic/01_dataset/spin_dynamics_phase_transition.png')


## 10. dnn_synthetic/02_complexity_measure/beta_complexity_figure.png

Source: `02_dnn_synthetic/figures/02_complexity_measure/beta_complexity_figure.png`

In [ ]:
build_single_figure('dnn_synthetic/02_complexity_measure/beta_complexity_figure.png')


## 11. dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p05.png

Source: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/cell_beta_0p05.png`

In [ ]:
build_single_figure('dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p05.png')


## 12. dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p07.png

Source: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/cell_beta_0p07.png`

In [ ]:
build_single_figure('dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p07.png')


## 13. dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p09.png

Source: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/cell_beta_0p09.png`

In [ ]:
build_single_figure('dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p09.png')


## 14. dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p11.png

Source: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/cell_beta_0p11.png`

In [ ]:
build_single_figure('dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p11.png')


## 15. dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p13.png

Source: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/cell_beta_0p13.png`

In [ ]:
build_single_figure('dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p13.png')


## 16. dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p15.png

Source: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/cell_beta_0p15.png`

In [ ]:
build_single_figure('dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p15.png')


## 17. dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p17.png

Source: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/cell_beta_0p17.png`

In [ ]:
build_single_figure('dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p17.png')


## 18. dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p19.png

Source: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/cell_beta_0p19.png`

In [ ]:
build_single_figure('dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p19.png')


## 19. dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p21.png

Source: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/cell_beta_0p21.png`

In [ ]:
build_single_figure('dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p21.png')


## 20. dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p23.png

Source: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/cell_beta_0p23.png`

In [ ]:
build_single_figure('dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p23.png')


## 21. dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p25.png

Source: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/cell_beta_0p25.png`

In [ ]:
build_single_figure('dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p25.png')


## 22. dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p27.png

Source: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/cell_beta_0p27.png`

In [ ]:
build_single_figure('dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p27.png')


## 23. dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p29.png

Source: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/cell_beta_0p29.png`

In [ ]:
build_single_figure('dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p29.png')


## 24. dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p31.png

Source: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/cell_beta_0p31.png`

In [ ]:
build_single_figure('dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p31.png')


## 25. dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p33.png

Source: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/cell_beta_0p33.png`

In [ ]:
build_single_figure('dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p33.png')


## 26. dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p35.png

Source: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/cell_beta_0p35.png`

In [ ]:
build_single_figure('dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p35.png')


## 27. dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p37.png

Source: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/cell_beta_0p37.png`

In [ ]:
build_single_figure('dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p37.png')


## 28. dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p39.png

Source: `02_dnn_synthetic/figures/04_sampling/logZ_split_distributions/cell_beta_0p39.png`

In [ ]:
build_single_figure('dnn_synthetic/04_sampling/logZ_split_distributions/cell_beta_0p39.png')


## 29. dnn_synthetic/05_proxy_local_entropy/derivative_phi_d_curve.png

Source: `02_dnn_synthetic/figures/05_proxy_local_entropy/derivative_phi_d_curve.png`

In [ ]:
build_single_figure('dnn_synthetic/05_proxy_local_entropy/derivative_phi_d_curve.png')


## 30. dnn_synthetic/05_proxy_local_entropy/derivative_phi_energetic_d_curve.png

Source: `02_dnn_synthetic/figures/05_proxy_local_entropy/derivative_phi_energetic_d_curve.png`

In [ ]:
build_single_figure('dnn_synthetic/05_proxy_local_entropy/derivative_phi_energetic_d_curve.png')


## 31. dnn_synthetic/05_proxy_local_entropy/phase_like_A_by_beta.png

Source: `02_dnn_synthetic/figures/05_proxy_local_entropy/phase_like_A_by_beta.png`

In [ ]:
build_single_figure('dnn_synthetic/05_proxy_local_entropy/phase_like_A_by_beta.png')


## 32. dnn_synthetic/05_proxy_local_entropy/phase_like_A_by_complexity.png

Source: `02_dnn_synthetic/figures/05_proxy_local_entropy/phase_like_A_by_complexity.png`

In [ ]:
build_single_figure('dnn_synthetic/05_proxy_local_entropy/phase_like_A_by_complexity.png')


## 33. dnn_synthetic/05_proxy_local_entropy/phi_d_curve.png

Source: `02_dnn_synthetic/figures/05_proxy_local_entropy/phi_d_curve.png`

In [ ]:
build_single_figure('dnn_synthetic/05_proxy_local_entropy/phi_d_curve.png')


## 34. dnn_synthetic/05_proxy_local_entropy/phi_energetic_d_curve.png

Source: `02_dnn_synthetic/figures/05_proxy_local_entropy/phi_energetic_d_curve.png`

In [ ]:
build_single_figure('dnn_synthetic/05_proxy_local_entropy/phi_energetic_d_curve.png')


## 35. dnn_mnist/01_dataset/label_noise_sweep/sample_figure.png

Source: `03_dnn_mnist/figures/01_dataset/label_noise_sweep/sample_figure.png`

In [ ]:
build_single_figure('dnn_mnist/01_dataset/label_noise_sweep/sample_figure.png')


## 36. dnn_mnist/01_dataset/manual_rules/sample_figure.png

Source: `03_dnn_mnist/figures/01_dataset/manual_rules/sample_figure.png`

In [ ]:
build_single_figure('dnn_mnist/01_dataset/manual_rules/sample_figure.png')


## 37. dnn_mnist/02_complexity_measure/label_noise_sweep/eta_complexity_figure.png

Source: `03_dnn_mnist/figures/02_complexity_measure/label_noise_sweep/eta_complexity_figure.png`

In [ ]:
build_single_figure('dnn_mnist/02_complexity_measure/label_noise_sweep/eta_complexity_figure.png')


## 38. dnn_mnist/02_complexity_measure/manual_rules/manual_rule_complexity_figure.png

Source: `03_dnn_mnist/figures/02_complexity_measure/manual_rules/manual_rule_complexity_figure.png`

In [ ]:
build_single_figure('dnn_mnist/02_complexity_measure/manual_rules/manual_rule_complexity_figure.png')


## 39. dnn_mnist/03_reference_search/label_noise_sweep/reference_quality_by_eta.png

Source: `03_dnn_mnist/figures/03_reference_search/label_noise_sweep/reference_quality_by_eta.png`

In [ ]:
build_single_figure('dnn_mnist/03_reference_search/label_noise_sweep/reference_quality_by_eta.png')


## 40. dnn_mnist/03_reference_search/manual_rules/reference_quality_by_rule.png

Source: `03_dnn_mnist/figures/03_reference_search/manual_rules/reference_quality_by_rule.png`

In [ ]:
build_single_figure('dnn_mnist/03_reference_search/manual_rules/reference_quality_by_rule.png')


## 41. dnn_mnist/04_sampling/label_noise_sweep/eta_reference_direct_derivative.png

Source: `03_dnn_mnist/figures/04_sampling/label_noise_sweep/eta_reference_direct_derivative.png`

In [ ]:
build_single_figure('dnn_mnist/04_sampling/label_noise_sweep/eta_reference_direct_derivative.png')


## 42. dnn_mnist/04_sampling/label_noise_sweep/eta_reference_phi_energy.png

Source: `03_dnn_mnist/figures/04_sampling/label_noise_sweep/eta_reference_phi_energy.png`

In [ ]:
build_single_figure('dnn_mnist/04_sampling/label_noise_sweep/eta_reference_phi_energy.png')


## 43. dnn_mnist/04_sampling/label_noise_sweep/logZ_split_distributions/noise_eta_0p05.png

Source: `03_dnn_mnist/figures/04_sampling/label_noise_sweep/logZ_split_distributions/noise_eta_0p05.png`

In [ ]:
build_single_figure('dnn_mnist/04_sampling/label_noise_sweep/logZ_split_distributions/noise_eta_0p05.png')


## 44. dnn_mnist/04_sampling/label_noise_sweep/logZ_split_distributions/noise_eta_0p15.png

Source: `03_dnn_mnist/figures/04_sampling/label_noise_sweep/logZ_split_distributions/noise_eta_0p15.png`

In [ ]:
build_single_figure('dnn_mnist/04_sampling/label_noise_sweep/logZ_split_distributions/noise_eta_0p15.png')


## 45. dnn_mnist/04_sampling/label_noise_sweep/logZ_split_distributions/noise_eta_0p25.png

Source: `03_dnn_mnist/figures/04_sampling/label_noise_sweep/logZ_split_distributions/noise_eta_0p25.png`

In [ ]:
build_single_figure('dnn_mnist/04_sampling/label_noise_sweep/logZ_split_distributions/noise_eta_0p25.png')


## 46. dnn_mnist/04_sampling/manual_rules/fresh108_validation/logZ_split_distributions/random_label.png

Source: `03_dnn_mnist/figures/04_sampling/manual_rules/fresh108_validation/logZ_split_distributions/random_label.png`

In [ ]:
build_single_figure('dnn_mnist/04_sampling/manual_rules/fresh108_validation/logZ_split_distributions/random_label.png')


## 47. dnn_mnist/04_sampling/manual_rules/fresh108_validation/logZ_split_distributions/real_even_odd.png

Source: `03_dnn_mnist/figures/04_sampling/manual_rules/fresh108_validation/logZ_split_distributions/real_even_odd.png`

In [ ]:
build_single_figure('dnn_mnist/04_sampling/manual_rules/fresh108_validation/logZ_split_distributions/real_even_odd.png')


## 48. dnn_mnist/04_sampling/manual_rules/fresh108_validation/logZ_split_distributions/teacher_nn.png

Source: `03_dnn_mnist/figures/04_sampling/manual_rules/fresh108_validation/logZ_split_distributions/teacher_nn.png`

In [ ]:
build_single_figure('dnn_mnist/04_sampling/manual_rules/fresh108_validation/logZ_split_distributions/teacher_nn.png')


## 49. dnn_mnist/04_sampling/manual_rules/fresh108_validation/logZ_split_distributions/very_low_tv_spectral_teacher.png

Source: `03_dnn_mnist/figures/04_sampling/manual_rules/fresh108_validation/logZ_split_distributions/very_low_tv_spectral_teacher.png`

In [ ]:
build_single_figure('dnn_mnist/04_sampling/manual_rules/fresh108_validation/logZ_split_distributions/very_low_tv_spectral_teacher.png')


## 50. dnn_mnist/04_sampling/manual_rules/fresh108_validation/manual_rule_direct_derivative.png

Source: `03_dnn_mnist/figures/04_sampling/manual_rules/fresh108_validation/manual_rule_direct_derivative.png`

In [ ]:
build_single_figure('dnn_mnist/04_sampling/manual_rules/fresh108_validation/manual_rule_direct_derivative.png')


## 51. dnn_mnist/04_sampling/manual_rules/fresh108_validation/manual_rule_ess_q05.png

Source: `03_dnn_mnist/figures/04_sampling/manual_rules/fresh108_validation/manual_rule_ess_q05.png`

In [ ]:
build_single_figure('dnn_mnist/04_sampling/manual_rules/fresh108_validation/manual_rule_ess_q05.png')


## 52. dnn_mnist/04_sampling/manual_rules/fresh108_validation/manual_rule_logZ_split_heatmap.png

Source: `03_dnn_mnist/figures/04_sampling/manual_rules/fresh108_validation/manual_rule_logZ_split_heatmap.png`

In [ ]:
build_single_figure('dnn_mnist/04_sampling/manual_rules/fresh108_validation/manual_rule_logZ_split_heatmap.png')


## 53. dnn_mnist/04_sampling/manual_rules/fresh108_validation/manual_rule_phi_energy.png

Source: `03_dnn_mnist/figures/04_sampling/manual_rules/fresh108_validation/manual_rule_phi_energy.png`

In [ ]:
build_single_figure('dnn_mnist/04_sampling/manual_rules/fresh108_validation/manual_rule_phi_energy.png')


## 54. dnn_mnist/04_sampling/manual_rules/logZ_split_distributions/random_label.png

Source: `03_dnn_mnist/figures/04_sampling/manual_rules/logZ_split_distributions/random_label.png`

In [ ]:
build_single_figure('dnn_mnist/04_sampling/manual_rules/logZ_split_distributions/random_label.png')


## 55. dnn_mnist/04_sampling/manual_rules/logZ_split_distributions/real_even_odd.png

Source: `03_dnn_mnist/figures/04_sampling/manual_rules/logZ_split_distributions/real_even_odd.png`

In [ ]:
build_single_figure('dnn_mnist/04_sampling/manual_rules/logZ_split_distributions/real_even_odd.png')


## 56. dnn_mnist/04_sampling/manual_rules/logZ_split_distributions/teacher_nn.png

Source: `03_dnn_mnist/figures/04_sampling/manual_rules/logZ_split_distributions/teacher_nn.png`

In [ ]:
build_single_figure('dnn_mnist/04_sampling/manual_rules/logZ_split_distributions/teacher_nn.png')


## 57. dnn_mnist/04_sampling/manual_rules/logZ_split_distributions/very_low_tv_spectral_teacher.png

Source: `03_dnn_mnist/figures/04_sampling/manual_rules/logZ_split_distributions/very_low_tv_spectral_teacher.png`

In [ ]:
build_single_figure('dnn_mnist/04_sampling/manual_rules/logZ_split_distributions/very_low_tv_spectral_teacher.png')


## 58. dnn_mnist/04_sampling/manual_rules/manual_rule_direct_derivative.png

Source: `03_dnn_mnist/figures/04_sampling/manual_rules/manual_rule_direct_derivative.png`

In [ ]:
build_single_figure('dnn_mnist/04_sampling/manual_rules/manual_rule_direct_derivative.png')


## 59. dnn_mnist/04_sampling/manual_rules/manual_rule_ess_q05.png

Source: `03_dnn_mnist/figures/04_sampling/manual_rules/manual_rule_ess_q05.png`

In [ ]:
build_single_figure('dnn_mnist/04_sampling/manual_rules/manual_rule_ess_q05.png')


## 60. dnn_mnist/04_sampling/manual_rules/manual_rule_logZ_split_heatmap.png

Source: `03_dnn_mnist/figures/04_sampling/manual_rules/manual_rule_logZ_split_heatmap.png`

In [ ]:
build_single_figure('dnn_mnist/04_sampling/manual_rules/manual_rule_logZ_split_heatmap.png')


## 61. dnn_mnist/04_sampling/manual_rules/manual_rule_phi_energy.png

Source: `03_dnn_mnist/figures/04_sampling/manual_rules/manual_rule_phi_energy.png`

In [ ]:
build_single_figure('dnn_mnist/04_sampling/manual_rules/manual_rule_phi_energy.png')


## 62. dnn_mnist/05_proxy_local_entropy/label_noise_sweep/derivative_phi_d_curve.png

Source: `03_dnn_mnist/figures/05_proxy_local_entropy/label_noise_sweep/derivative_phi_d_curve.png`

In [ ]:
build_single_figure('dnn_mnist/05_proxy_local_entropy/label_noise_sweep/derivative_phi_d_curve.png')


## 63. dnn_mnist/05_proxy_local_entropy/label_noise_sweep/derivative_phi_energetic_d_curve.png

Source: `03_dnn_mnist/figures/05_proxy_local_entropy/label_noise_sweep/derivative_phi_energetic_d_curve.png`

In [ ]:
build_single_figure('dnn_mnist/05_proxy_local_entropy/label_noise_sweep/derivative_phi_energetic_d_curve.png')


## 64. dnn_mnist/05_proxy_local_entropy/label_noise_sweep/logZ_split_qc_results.png

Source: `03_dnn_mnist/figures/05_proxy_local_entropy/label_noise_sweep/logZ_split_qc_results.png`

In [ ]:
build_single_figure('dnn_mnist/05_proxy_local_entropy/label_noise_sweep/logZ_split_qc_results.png')


## 65. dnn_mnist/05_proxy_local_entropy/label_noise_sweep/phase_like_A_by_complexity.png

Source: `03_dnn_mnist/figures/05_proxy_local_entropy/label_noise_sweep/phase_like_A_by_complexity.png`

In [ ]:
build_single_figure('dnn_mnist/05_proxy_local_entropy/label_noise_sweep/phase_like_A_by_complexity.png')


## 66. dnn_mnist/05_proxy_local_entropy/label_noise_sweep/phase_like_A_by_eta.png

Source: `03_dnn_mnist/figures/05_proxy_local_entropy/label_noise_sweep/phase_like_A_by_eta.png`

In [ ]:
build_single_figure('dnn_mnist/05_proxy_local_entropy/label_noise_sweep/phase_like_A_by_eta.png')


## 67. dnn_mnist/05_proxy_local_entropy/label_noise_sweep/phi_d_curve.png

Source: `03_dnn_mnist/figures/05_proxy_local_entropy/label_noise_sweep/phi_d_curve.png`

In [ ]:
build_single_figure('dnn_mnist/05_proxy_local_entropy/label_noise_sweep/phi_d_curve.png')


## 68. dnn_mnist/05_proxy_local_entropy/label_noise_sweep/phi_energetic_d_curve.png

Source: `03_dnn_mnist/figures/05_proxy_local_entropy/label_noise_sweep/phi_energetic_d_curve.png`

In [ ]:
build_single_figure('dnn_mnist/05_proxy_local_entropy/label_noise_sweep/phi_energetic_d_curve.png')


## 69. dnn_mnist/05_proxy_local_entropy/label_noise_sweep/reference_variability_results.png

Source: `03_dnn_mnist/figures/05_proxy_local_entropy/label_noise_sweep/reference_variability_results.png`

In [ ]:
build_single_figure('dnn_mnist/05_proxy_local_entropy/label_noise_sweep/reference_variability_results.png')


## 70. dnn_mnist/05_proxy_local_entropy/manual_rules/derivative_phi_d_curve.png

Source: `03_dnn_mnist/figures/05_proxy_local_entropy/manual_rules/derivative_phi_d_curve.png`

In [ ]:
build_single_figure('dnn_mnist/05_proxy_local_entropy/manual_rules/derivative_phi_d_curve.png')


## 71. dnn_mnist/05_proxy_local_entropy/manual_rules/derivative_phi_energetic_d_curve.png

Source: `03_dnn_mnist/figures/05_proxy_local_entropy/manual_rules/derivative_phi_energetic_d_curve.png`

In [ ]:
build_single_figure('dnn_mnist/05_proxy_local_entropy/manual_rules/derivative_phi_energetic_d_curve.png')


## 72. dnn_mnist/05_proxy_local_entropy/manual_rules/logZ_split_qc_results.png

Source: `03_dnn_mnist/figures/05_proxy_local_entropy/manual_rules/logZ_split_qc_results.png`

In [ ]:
build_single_figure('dnn_mnist/05_proxy_local_entropy/manual_rules/logZ_split_qc_results.png')


## 73. dnn_mnist/05_proxy_local_entropy/manual_rules/phase_like_A_by_complexity.png

Source: `03_dnn_mnist/figures/05_proxy_local_entropy/manual_rules/phase_like_A_by_complexity.png`

In [ ]:
build_single_figure('dnn_mnist/05_proxy_local_entropy/manual_rules/phase_like_A_by_complexity.png')


## 74. dnn_mnist/05_proxy_local_entropy/manual_rules/phase_like_A_by_rule.png

Source: `03_dnn_mnist/figures/05_proxy_local_entropy/manual_rules/phase_like_A_by_rule.png`

In [ ]:
build_single_figure('dnn_mnist/05_proxy_local_entropy/manual_rules/phase_like_A_by_rule.png')


## 75. dnn_mnist/05_proxy_local_entropy/manual_rules/phi_d_curve.png

Source: `03_dnn_mnist/figures/05_proxy_local_entropy/manual_rules/phi_d_curve.png`

In [ ]:
build_single_figure('dnn_mnist/05_proxy_local_entropy/manual_rules/phi_d_curve.png')


## 76. dnn_mnist/05_proxy_local_entropy/manual_rules/phi_energetic_d_curve.png

Source: `03_dnn_mnist/figures/05_proxy_local_entropy/manual_rules/phi_energetic_d_curve.png`

In [ ]:
build_single_figure('dnn_mnist/05_proxy_local_entropy/manual_rules/phi_energetic_d_curve.png')


## 77. dnn_mnist/05_proxy_local_entropy/manual_rules/reference_variability_results.png

Source: `03_dnn_mnist/figures/05_proxy_local_entropy/manual_rules/reference_variability_results.png`

In [ ]:
build_single_figure('dnn_mnist/05_proxy_local_entropy/manual_rules/reference_variability_results.png')
